In [1]:
from pyspark.sql.functions import col, row_number, coalesce, max as spark_max, lit
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# 1. Read the cleaned Silver data
silver_df = spark.read.table("silver_songs_metadata")

# -------------------------------------------------------------------------
# 2. Dim_Category (Incremental Surrogate Key Assignment)
# -------------------------------------------------------------------------
categories_source = silver_df.select("Main_Category", "Sub_Category").distinct()

if spark.catalog.tableExists("Dim_Category"):
    existing_cat = spark.read.table("Dim_Category")
    
    # Identify only truly new categories by doing a Left Anti join
    new_categories = categories_source.join(existing_cat, on=["Main_Category", "Sub_Category"], how="left_anti")
    
    if not new_categories.isEmpty():
        max_cat_id = existing_cat.select(coalesce(spark_max("Category_ID"), lit(0))).collect()[0][0]
        window_cat = Window.orderBy("Main_Category", "Sub_Category")
        new_categories_with_id = new_categories.withColumn("Category_ID", row_number().over(window_cat) + max_cat_id)
        
        new_categories_with_id.write.format("delta").mode("append").saveAsTable("Dim_Category")
        print(f"Added {new_categories.count()} new categories to Dim_Category.")
    else:
        print("No new categories found. Dim_Category is up to date.")
else:
    # First-time creation
    window_cat = Window.orderBy("Main_Category", "Sub_Category")
    dim_category = categories_source.withColumn("Category_ID", row_number().over(window_cat))
    dim_category.write.format("delta").mode("overwrite").saveAsTable("Dim_Category")
    print("Dim_Category created.")

# -------------------------------------------------------------------------
# 3. Dim_Scale (Incremental Surrogate Key Assignment)
# -------------------------------------------------------------------------
scales_source = silver_df.select("Scale").distinct().filter(col("Scale").isNotNull())

if spark.catalog.tableExists("Dim_Scale"):
    existing_scale = spark.read.table("Dim_Scale")
    
    new_scales = scales_source.join(existing_scale, on="Scale", how="left_anti")
    
    if not new_scales.isEmpty():
        max_scale_id = existing_scale.select(coalesce(spark_max("Scale_ID"), lit(0))).collect()[0][0]
        window_scale = Window.orderBy("Scale")
        new_scales_with_id = new_scales.withColumn("Scale_ID", row_number().over(window_scale) + max_scale_id)
        
        new_scales_with_id.write.format("delta").mode("append").saveAsTable("Dim_Scale")
        print(f"Added {new_scales.count()} new scales to Dim_Scale.")
    else:
         print("No new scales found. Dim_Scale is up to date.")
else:
    window_scale = Window.orderBy("Scale")
    dim_scale = scales_source.withColumn("Scale_ID", row_number().over(window_scale))
    dim_scale.write.format("delta").mode("overwrite").saveAsTable("Dim_Scale")
    print("Dim_Scale created.")

# -------------------------------------------------------------------------
# 4. Fact_Songs (Merge Upsert)
# -------------------------------------------------------------------------
# Reload up-to-date dimensions to ensure new IDs are caught for the join
dim_category_upd = spark.read.table("Dim_Category")
dim_scale_upd = spark.read.table("Dim_Scale")

fact_source = silver_df.join(dim_category_upd, on=["Main_Category", "Sub_Category"], how="left") \
                       .join(dim_scale_upd, on="Scale", how="left")

columns_to_select = [
    "Category_ID", "Scale_ID", "Title", "Language", 
    "Singer", "Composer", "Raag", "Movie", "Album", "Poet", 
    "Page_No", "Source_Path"
]
actual_columns = [c for c in columns_to_select if c in fact_source.columns]
fact_source = fact_source.select(*actual_columns)

if spark.catalog.tableExists("Fact_Songs"):
    fact_target = DeltaTable.forName(spark, "Fact_Songs")
    fact_target.alias("target") \
        .merge(
            fact_source.alias("source"),
            "target.Title = source.Title AND target.Source_Path = source.Source_Path"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Fact_Songs successfully merged with new data.")
else:
    fact_source.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Fact_Songs")
    print("Fact_Songs created and loaded.")
    
# Release compute cluster resources
spark.stop()

StatementMeta(, a77597ee-4935-464d-a4b7-e7ffc298c9a5, 3, Finished, Available, Finished, False)

No new categories found. Dim_Category is up to date.
No new scales found. Dim_Scale is up to date.
Fact_Songs successfully merged with new data.


In [ ]:
# Instantly release Spark compute resources to prevent pipeline capacity errors
spark.stop()